# 05 — Statistics (bootstrap CIs, paired tests vs paper 1) + complexity / latency
Needs: notebook 01 (internal predictions), notebook 03 (external predictions). Paper-1 prediction CSVs are optional (paths in the COMMON block);
if present, McNemar + paired bootstrap against the published OralVision-M4 are computed on the *same* images.

In [1]:
# ---- COMMON ----
# =============================================================================
# OralVision-P2  |  COMMON BLOCK (identical in every notebook)
# A Self-Supervised Vision Transformer Framework for Multi-Class Oral Disease Recognition
# DINOv2 ViT-S/14 + LoRA + MLP/cosine head + logit-adjusted loss (fp16)
# =============================================================================
import os
# Your env also has TensorFlow (broken protobuf). Tell transformers to ignore TF/Flax BEFORE it is imported.
os.environ["USE_TF"] = "0"; os.environ["USE_FLAX"] = "0"; os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"; os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")

import gc, re, csv, json, math, time, random, copy, hashlib, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torchvision
from torchvision import transforms
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score,
                             cohen_kappa_score, classification_report)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

# =============================================================================
# 1. PATHS  (EDIT ONLY THIS BLOCK IF YOUR FOLDERS DIFFER)
# =============================================================================
DESKTOP = Path(r"C:\Users\R Y Z E N\Desktop")

# --- same data as paper 1 (unchanged) ---
DATASET_ROOT     = DESKTOP / "oral_dataset_FINAL_CLEAN"        # contains train / val / test
RAW_DATA_DIR     = DESKTOP / "roboflow_dataset_merged"         # external, raw
BENGRAM_DATA_DIR = DESKTOP / "roboflow_bengram_illumination"   # external, Ben Graham

# --- all paper-2 outputs go here ---
P2_ROOT   = DESKTOP / "OralVision_P2"
OUT_00    = P2_ROOT / "00_setup_audit"
OUT_MAIN  = P2_ROOT / "01_main_5seed"
OUT_ABL   = P2_ROOT / "02_ablation"
OUT_EXT   = P2_ROOT / "03_external"
OUT_XAI   = P2_ROOT / "04_explainability"
OUT_STATS = P2_ROOT / "05_stats_complexity"

# --- paper-1 prediction files (only used by notebook 05; optional) ---
def _first_existing(*cands):
    for c in cands:
        if Path(c).exists():
            return Path(c)
    return Path(cands[0])

M4_INTERNAL_CSV = _first_existing(DESKTOP / "m4_statistical_rigor_report" / "test_predictions_all_models.csv",
                                  DESKTOP / "paper" / "m4_statistical_rigor_report" / "test_predictions_all_models.csv")
M4_EXT_RAW_CSV  = _first_existing(DESKTOP / "external_eval_results_CLEAN_MODEL" / "external_eval_raw" / "predictions.csv",
                                  DESKTOP / "paper" / "external_eval_results_CLEAN_MODEL" / "external_eval_raw" / "predictions.csv")
M4_EXT_BG_CSV   = _first_existing(DESKTOP / "external_eval_results_CLEAN_MODEL" / "external_eval_bengram" / "predictions.csv",
                                  DESKTOP / "paper" / "external_eval_results_CLEAN_MODEL" / "external_eval_bengram" / "predictions.csv")

for _d in (P2_ROOT, OUT_00, OUT_MAIN, OUT_ABL, OUT_EXT, OUT_XAI, OUT_STATS):
    _d.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 2. CONFIG
# =============================================================================
CLASS_NAMES = ["Calculus", "Caries", "Gingivitis", "Hypodontia", "Mouth_Ulcer", "Tooth_Discoloration"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
SEEDS = [42, 123, 2024, 3407, 7777]          # same five seeds as paper 1
PRIMARY_SEED = 42

BACKBONES = {
    "dinov2_s14":  "facebook/dinov2-small",              # self-supervised (proposed)
    "vit_s16_sup": "WinKawaks/vit-small-patch16-224",    # supervised ImageNet ViT-S/16 (ablation)
    "deit_s16_sup": "facebook/deit-small-patch16-224",   # fallback supervised ViT-S/16
}

CFG = dict(
    img_size=224,
    batch_size=16, grad_accum=2,              # effective 32, same as paper 1 (RTX 2050 4 GB safe)
    num_workers=0, pin_memory=True,           # keep 0 on Windows/Jupyter
    backbone="dinov2_s14",
    use_lora=True, lora_r=8, lora_alpha=16, lora_dropout=0.0,
    lora_targets=("query", "key", "value", "out"),
    head_type="mlp",                          # "mlp" | "cosine" | "linear"
    head_hidden=512, head_dropout=0.1, cos_scale=30.0,
    loss="la",                                # "la" logit-adjusted | "ce" plain | "cw" class-weighted (paper 1 style)
    la_tau=1.0, label_smoothing=0.0,
    aug="photometric",                        # "photometric" | "basic" (paper 1 style)
    epochs=15, warmup_epochs=1, patience=5,
    lr_lora=3e-4, lr_head=1e-3, weight_decay=0.05, clip_grad=1.0,
    use_amp=True,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass


def gpu_report(prefix=""):
    if not torch.cuda.is_available():
        return
    free, total = torch.cuda.mem_get_info()
    log(f"{prefix}GPU alloc={torch.cuda.memory_allocated()/1e9:.2f} GB | "
        f"peak={torch.cuda.max_memory_allocated()/1e9:.2f} GB | free={free/1e9:.2f}/{total/1e9:.2f} GB")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))


# =============================================================================
# 3. DATA  (identical split discovery / sample ordering as paper 1)
# =============================================================================
def find_split_directory(root, names):
    if isinstance(names, str):
        names = [names]
    lowered = {x.lower() for x in names}
    cands = []
    for p in Path(root).rglob("*"):
        if p.is_dir() and p.name.lower() in lowered:
            class_dirs = [x for x in p.iterdir() if x.is_dir()]
            if len(class_dirs) >= 2:
                cands.append((len(class_dirs), p))
    if not cands:
        return None
    cands.sort(key=lambda x: x[0], reverse=True)
    return cands[0][1]


def collect_samples(split_dir):
    samples = []
    for cname in CLASS_NAMES:
        cdir = Path(split_dir) / cname
        for path in sorted(cdir.rglob("*")):
            if path.is_file() and path.suffix.lower() in VALID_EXT:
                samples.append((str(path), CLASS_NAMES.index(cname)))
    return samples


def get_internal_splits():
    tr = find_split_directory(DATASET_ROOT, "train")
    va = find_split_directory(DATASET_ROOT, ["val", "validation"])
    te = find_split_directory(DATASET_ROOT, "test")
    if tr is None or va is None or te is None:
        raise FileNotFoundError(f"train/val/test folders not found under {DATASET_ROOT}")
    found = sorted(p.name for p in tr.iterdir() if p.is_dir())
    if found != sorted(CLASS_NAMES):
        raise RuntimeError(f"Class folders differ from expected.\nfound={found}\nexpected={sorted(CLASS_NAMES)}")
    return tr, va, te


class ImgDS(Dataset):
    """returns (image, label, index)"""
    def __init__(self, samples, transform):
        self.samples, self.transform = samples, transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i][0], self.samples[i][1]
        with Image.open(path) as im:
            im = im.convert("RGB")
        return self.transform(im), label, i


def collect_external_samples(root):
    """Same folder-name normalisation / duplicate-folder handling as paper 1's ExternalImageFolder.
    Returns list of (path, label, filename)."""
    canon = {c.lower(): c for c in CLASS_NAMES}
    seen, samples = {}, []
    for entry in sorted(os.listdir(root)):
        full = os.path.join(root, entry)
        if not os.path.isdir(full):
            continue
        c = canon.get(entry.strip().replace(" ", "_").lower())
        if c is None:
            log(f"  [WARN] unrecognised external folder '{entry}' skipped")
            continue
        if c in seen:
            log(f"  [NOTE] duplicate folder for '{c}' ('{entry}' vs '{seen[c]}') -> skipped duplicate")
            continue
        seen[c] = entry
        for fn in sorted(os.listdir(full)):
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                samples.append((os.path.join(full, fn), CLASS_NAMES.index(c), fn))
    if not samples:
        raise RuntimeError(f"No images found under {root}")
    return samples


def _has_classes(d):
    canon = {c.lower() for c in CLASS_NAMES}
    try:
        sub = [x.strip().replace(" ", "_").lower() for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))]
    except OSError:
        return False
    return sum(x in canon for x in sub) >= 3


def resolve_external_root(root, depth=3):
    """Finds the folder that directly contains the class sub-folders (handles nested folders of the same name)."""
    root = str(root)
    if not os.path.isdir(root):
        return None
    if _has_classes(root):
        return root
    if depth == 0:
        return None
    for x in sorted(os.listdir(root)):
        p = os.path.join(root, x)
        if os.path.isdir(p):
            r = resolve_external_root(p, depth - 1)
            if r:
                return r
    return None


def _resolve_first(*cands):
    for c in cands:
        r = resolve_external_root(c)
        if r:
            return Path(r)
    raise FileNotFoundError("External dataset not found. Tried:\n  " + "\n  ".join(str(c) for c in cands) +
                            "\nEdit RAW_DATA_DIR / BENGRAM_DATA_DIR at the top of this cell.")


RAW_DATA_DIR = _resolve_first(RAW_DATA_DIR, DESKTOP / "paper" / "roboflow_dataset_merged")
BENGRAM_DATA_DIR = _resolve_first(BENGRAM_DATA_DIR, DESKTOP / "paper" / "roboflow_bengram_illumination")
print(f"[paths] external raw     : {RAW_DATA_DIR}\n[paths] external BenGraham: {BENGRAM_DATA_DIR}")


# ----------------------------- augmentation ---------------------------------
class RandomColorConstancy:
    """Random re-illumination via Gray-World (p=1) / Shades-of-Gray (p=6) gains with random strength."""
    def __init__(self, p=0.5, minkowski=(1, 6), strength=(0.3, 1.0)):
        self.p, self.minkowski, self.strength = p, minkowski, strength

    def __call__(self, img):
        if random.random() > self.p:
            return img
        a = np.asarray(img, dtype=np.float32) / 255.0
        pn = random.choice(self.minkowski)
        e = np.power(np.power(a + 1e-6, pn).mean(axis=(0, 1)), 1.0 / pn)
        e = e / (np.linalg.norm(e) + 1e-8)
        gain = np.clip(1.0 / (e * math.sqrt(3.0) + 1e-8), 0.5, 2.0)
        gain = 1.0 + random.uniform(*self.strength) * (gain - 1.0)
        out = np.clip(a * gain, 0.0, 1.0)
        return Image.fromarray((out * 255.0).astype(np.uint8))


class RandomIlluminantShift:
    """Per-channel gain jitter (white-balance / illuminant colour change)."""
    def __init__(self, p=0.5, max_shift=0.12):
        self.p, self.m = p, max_shift

    def __call__(self, img):
        if random.random() > self.p:
            return img
        g = 1.0 + np.random.uniform(-self.m, self.m, size=3).astype(np.float32)
        a = np.asarray(img, dtype=np.float32) * g
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))


class RandomGamma:
    def __init__(self, p=0.4, rng=(0.7, 1.5)):
        self.p, self.rng = p, rng

    def __call__(self, img):
        if random.random() > self.p:
            return img
        g = random.uniform(*self.rng)
        a = (np.asarray(img, dtype=np.float32) / 255.0) ** g
        return Image.fromarray((np.clip(a, 0, 1) * 255.0).astype(np.uint8))


def build_train_transform(kind="photometric", size=224):
    norm = [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    if kind == "basic":      # paper-1 style
        return transforms.Compose([
            transforms.Resize((size, size)), transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)] + norm)
    return transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.75, 1.0), ratio=(0.9, 1.11)),
        transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
        RandomColorConstancy(0.5), RandomIlluminantShift(0.5), RandomGamma(0.4),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03),
        transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 1.5))], p=0.2)] + norm)


def build_eval_transform(size=224):
    return transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                               transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


def make_loader(ds, shuffle=False, batch_size=None, drop_last=False, generator=None):
    return DataLoader(ds, batch_size=batch_size or CFG["batch_size"], shuffle=shuffle,
                      num_workers=CFG["num_workers"], drop_last=drop_last, generator=generator,
                      pin_memory=(CFG["pin_memory"] and DEVICE.type == "cuda"))


# =============================================================================
# 4. MODEL: frozen ViT + LoRA + head
# =============================================================================
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base, self.r, self.scaling = base, r, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.lora_A.t() @ self.lora_B.t()) * self.scaling


_LORA_PATTERNS = {
    "query": re.compile(r"(^|\.)(query|q_proj)$"),
    "key":   re.compile(r"(^|\.)(key|k_proj)$"),
    "value": re.compile(r"(^|\.)(value|v_proj)$"),
    "out":   re.compile(r"(attention\.output\.dense|attn\.proj|attention\.out_proj|o_proj)$"),
}


def inject_lora(backbone, r, alpha, dropout, targets):
    to_wrap = []
    for name, mod in backbone.named_modules():
        if not isinstance(mod, nn.Linear):
            continue
        if ("attention" not in name) and ("attn" not in name):
            continue
        for t in targets:
            if _LORA_PATTERNS[t].search(name):
                to_wrap.append(name)
                break
    n_layers = backbone.config.num_hidden_layers
    if len(to_wrap) != n_layers * len(targets):
        raise RuntimeError(f"LoRA injection found {len(to_wrap)} linears, expected {n_layers*len(targets)}: {to_wrap[:8]}")
    for name in to_wrap:
        parent = backbone
        parts = name.split(".")
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], LoRALinear(getattr(parent, parts[-1]), r, alpha, dropout))
    return len(to_wrap)


def load_backbone(name, eager=False):
    from transformers import AutoModel
    kw = {"attn_implementation": "eager"} if eager else {}
    try:
        m = AutoModel.from_pretrained(BACKBONES[name], **kw)
    except Exception as e:
        if name == "vit_s16_sup":
            log(f"  [WARN] {BACKBONES[name]} failed ({type(e).__name__}); falling back to {BACKBONES['deit_s16_sup']}")
            m = AutoModel.from_pretrained(BACKBONES["deit_s16_sup"], **kw)
        else:
            raise
    return m


class CosineHead(nn.Module):
    def __init__(self, in_dim, num_classes, scale=30.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_dim) * 0.02)
        self.scale = nn.Parameter(torch.tensor(float(scale)))

    def forward(self, x):
        return self.scale * F.linear(F.normalize(x.float(), dim=-1), F.normalize(self.weight.float(), dim=-1))


def build_head(head_type, in_dim, num_classes, hidden, dropout, cos_scale):
    if head_type == "linear":
        return nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, num_classes))
    if head_type == "mlp":
        return nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, hidden), nn.GELU(),
                             nn.Dropout(dropout), nn.Linear(hidden, num_classes))
    if head_type == "cosine":
        return nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, hidden), nn.GELU(),
                             CosineHead(hidden, num_classes, cos_scale))
    raise ValueError(head_type)


class OralViT(nn.Module):
    """Backbone (frozen) [+ LoRA] -> [CLS ; mean-patch] -> head."""
    def __init__(self, cfg=None, eager=False):
        super().__init__()
        c = dict(CFG if cfg is None else cfg)
        self.cfg = c
        self.backbone = load_backbone(c["backbone"], eager=eager)
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.n_lora = 0
        if c["use_lora"]:
            self.n_lora = inject_lora(self.backbone, c["lora_r"], c["lora_alpha"], c["lora_dropout"],
                                      tuple(c["lora_targets"]))
        dim = self.backbone.config.hidden_size
        self.feat_dim = 2 * dim
        self.head = build_head(c["head_type"], self.feat_dim, NUM_CLASSES, c["head_hidden"],
                               c["head_dropout"], c["cos_scale"])

    def encode(self, x):
        if self.cfg["use_lora"]:
            h = self.backbone(pixel_values=x).last_hidden_state
        else:
            with torch.no_grad():
                h = self.backbone(pixel_values=x).last_hidden_state
        return torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=-1)

    def forward(self, x):
        return self.head(self.encode(x))


def trainable_state_dict(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


def load_model_from_ckpt(path, eager=False):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m = OralViT(ck["cfg"], eager=eager)
    missing, unexpected = m.load_state_dict(ck["trainable_state"], strict=False)
    bad = [k for k in missing if ("lora" in k or k.startswith("head"))]
    if bad or unexpected:
        raise RuntimeError(f"Checkpoint mismatch. missing(trainable)={bad[:5]} unexpected={unexpected[:5]}")
    return m.to(DEVICE).eval(), ck


# =============================================================================
# 5. LOSS, METRICS
# =============================================================================
class LogitAdjustedCE(nn.Module):
    """Menon et al. 2021: add tau*log(prior) to logits at TRAIN time only; plain logits at inference."""
    def __init__(self, class_counts, tau=1.0, label_smoothing=0.0):
        super().__init__()
        prior = torch.tensor(class_counts, dtype=torch.float32)
        prior = prior / prior.sum()
        self.register_buffer("adj", tau * torch.log(prior))
        self.ls = label_smoothing

    def forward(self, logits, y):
        return F.cross_entropy(logits.float() + self.adj, y, label_smoothing=self.ls)


def build_criterion(loss_name, train_labels):
    counts = np.bincount(train_labels, minlength=NUM_CLASSES)
    if loss_name == "la":
        return LogitAdjustedCE(counts, CFG["la_tau"], CFG["label_smoothing"]).to(DEVICE)
    if loss_name == "cw":
        from sklearn.utils.class_weight import compute_class_weight
        w = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_labels)
        return nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE),
                                   label_smoothing=CFG["label_smoothing"])
    return nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])


def expected_calibration_error(prob, y, n_bins=15):
    conf, pred = prob.max(1), prob.argmax(1)
    acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def compute_metrics(y, prob):
    prob = np.asarray(prob, dtype=np.float64)
    prob = prob / prob.sum(1, keepdims=True)
    pred = prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=list(range(NUM_CLASSES)),
                                                 average=None, zero_division=0)
    try:
        auc = float(roc_auc_score(label_binarize(y, classes=list(range(NUM_CLASSES))), prob,
                                  average="macro", multi_class="ovr"))
    except Exception:
        auc = float("nan")
    return dict(
        n=int(len(y)), accuracy=float(accuracy_score(y, pred)),
        balanced_accuracy=float(balanced_accuracy_score(y, pred)),
        macro_precision=float(p.mean()), macro_recall=float(r.mean()), macro_f1=float(f.mean()),
        macro_auc=auc, ece=expected_calibration_error(prob, y),
        qwk=float(cohen_kappa_score(y, pred, weights="quadratic")),
        per_class_precision=p.tolist(), per_class_recall=r.tolist(), per_class_f1=f.tolist(),
        tooth_disc_recall=float(r[CLASS_NAMES.index("Tooth_Discoloration")]),
        confusion=confusion_matrix(y, pred, labels=list(range(NUM_CLASSES))).tolist(),
    )


@torch.no_grad()
def predict_loader(model, loader, return_idx=False):
    model.eval()
    probs, ys, idxs = [], [], []
    for batch in loader:
        x, y = batch[0].to(DEVICE, non_blocking=True), batch[1]
        with autocast(device_type=DEVICE.type, enabled=CFG["use_amp"] and DEVICE.type == "cuda"):
            logits = model(x)
        probs.append(F.softmax(logits.float(), 1).cpu().numpy())
        ys.append(np.asarray(y))
        idxs.append(np.asarray(batch[2]))
    out = (np.concatenate(probs), np.concatenate(ys))
    return out + (np.concatenate(idxs),) if return_idx else out


def plot_confusion(cm, title, path, normalize=False):
    cm = np.array(cm)
    fig, ax = plt.subplots(figsize=(7.5, 6))
    data = cm / cm.sum(1, keepdims=True).clip(min=1) if normalize else cm
    sns.heatmap(data, annot=True, fmt=".2f" if normalize else "d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


def write_predictions_csv(path, samples, y, prob, with_path=True):
    pred = prob.argmax(1)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["image_index", "filename", "true_label", "pred_label", "correct"] +
                   [f"prob_{c}" for c in CLASS_NAMES])
        for i in range(len(y)):
            fn = os.path.basename(samples[i][0])
            w.writerow([i, fn, CLASS_NAMES[y[i]], CLASS_NAMES[pred[i]], int(y[i] == pred[i])] +
                       [f"{p:.6f}" for p in prob[i]])


# =============================================================================
# 6. TRAINING (resumable, per-epoch checkpoint)
# =============================================================================
def _run_training_impl(run_dir, seed, cfg_overrides=None, train_samples=None, val_samples=None, test_samples=None,
                 eval_train_at_end=False, tag=""):
    """Train one model. Skips if summary.json exists. Returns summary dict."""
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=True)
    sum_path = run_dir / "summary.json"
    if sum_path.exists():
        log(f"[skip] {run_dir.name} already finished")
        return json.load(open(sum_path))

    base_cfg = copy.deepcopy(CFG)
    if cfg_overrides:
        CFG.update(cfg_overrides)
    try:
        cfg = copy.deepcopy(CFG)
        seed_everything(seed)
        log("=" * 70); log(f"RUN {tag or run_dir.name} | seed={seed}"); log("=" * 70)
        log(f"cfg: " + json.dumps({k: cfg[k] for k in ('backbone', 'use_lora', 'lora_r', 'lora_targets', 'head_type',
                                                       'loss', 'aug', 'epochs')}, default=str))

        tr_ds = ImgDS(train_samples, build_train_transform(cfg["aug"], cfg["img_size"]))
        va_ds = ImgDS(val_samples, build_eval_transform(cfg["img_size"]))
        te_ds = ImgDS(test_samples, build_eval_transform(cfg["img_size"]))
        g = torch.Generator(); g.manual_seed(seed)
        tr_loader = make_loader(tr_ds, shuffle=True, drop_last=True, generator=g)
        va_loader, te_loader = make_loader(va_ds), make_loader(te_ds)
        train_labels = np.array([s[1] for s in train_samples])

        model = OralViT(cfg).to(DEVICE)
        tot, trn = count_params(model)
        log(f"params total={tot:,} trainable={trn:,} ({100*trn/tot:.2f}%)")
        criterion = build_criterion(cfg["loss"], train_labels)

        lora_p = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
        head_p = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n]
        groups = []
        if lora_p: groups.append(dict(params=lora_p, lr=cfg["lr_lora"], weight_decay=0.0))
        groups.append(dict(params=head_p, lr=cfg["lr_head"], weight_decay=cfg["weight_decay"]))
        opt = torch.optim.AdamW(groups)
        base_lrs = [gr["lr"] for gr in opt.param_groups]
        steps_per_epoch = len(tr_loader) // cfg["grad_accum"]
        total_steps = steps_per_epoch * cfg["epochs"]
        warm_steps = steps_per_epoch * cfg["warmup_epochs"]

        def set_lr(step):
            if step < warm_steps:
                f = (step + 1) / max(1, warm_steps)
            else:
                f = 0.5 * (1 + math.cos(math.pi * (step - warm_steps) / max(1, total_steps - warm_steps)))
            for gr, b in zip(opt.param_groups, base_lrs):
                gr["lr"] = b * max(f, 0.01)

        use_amp = cfg["use_amp"] and DEVICE.type == "cuda"
        scaler = GradScaler("cuda", enabled=use_amp)

        start_epoch, best_f1, best_ep, bad, hist, gstep = 0, -1.0, -1, 0, [], 0
        last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"
        if last_path.exists():
            ck = torch.load(last_path, map_location="cpu", weights_only=False)
            model.load_state_dict(ck["trainable_state"], strict=False)
            opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
            start_epoch, best_f1, best_ep, bad, hist, gstep = (ck["epoch"] + 1, ck["best_f1"], ck["best_ep"],
                                                               ck["bad"], ck["hist"], ck["gstep"])
            log(f"Resumed from epoch {start_epoch} (best macro-F1 so far {best_f1:.4f})")

        t_start = time.time()
        for ep in range(start_epoch, cfg["epochs"]):
            model.train()
            if not cfg["use_lora"]:
                model.backbone.eval()
            t0, run_loss, n_seen, n_corr = time.time(), 0.0, 0, 0
            opt.zero_grad(set_to_none=True)
            for it, (x, y, _) in enumerate(tr_loader):
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                with autocast(device_type=DEVICE.type, enabled=use_amp):
                    logits = model(x)
                loss = criterion(logits.float(), y)
                scaler.scale(loss / cfg["grad_accum"]).backward()
                if (it + 1) % cfg["grad_accum"] == 0:
                    set_lr(gstep)
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_([p for gr in opt.param_groups for p in gr["params"]], cfg["clip_grad"])
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                    gstep += 1
                run_loss += loss.item() * len(y); n_seen += len(y)
                n_corr += (logits.argmax(1) == y).sum().item()
                if it % 200 == 0:
                    log(f"  ep{ep+1} it{it}/{len(tr_loader)} loss={run_loss/max(n_seen,1):.4f}")
            vprob, vy = predict_loader(model, va_loader)
            vm = compute_metrics(vy, vprob)
            vloss = float(-np.log(np.clip(vprob[np.arange(len(vy)), vy], 1e-8, 1)).mean())
            rec = dict(epoch=ep + 1, train_loss=run_loss / n_seen, train_acc=n_corr / n_seen, val_loss=vloss,
                       val_acc=vm["accuracy"], val_macro_f1=vm["macro_f1"], val_bal_acc=vm["balanced_accuracy"],
                       lr_head=opt.param_groups[-1]["lr"], sec=time.time() - t0)
            hist.append(rec)
            log(f"ep {ep+1}/{cfg['epochs']} | train_loss {rec['train_loss']:.4f} acc {rec['train_acc']:.4f} | "
                f"val_loss {vloss:.4f} acc {vm['accuracy']:.4f} macroF1 {vm['macro_f1']:.4f} | {rec['sec']:.0f}s | GPU {gpu_temp_clock()}")
            if vm["macro_f1"] > best_f1:
                best_f1, best_ep, bad = vm["macro_f1"], ep + 1, 0
                torch.save(dict(trainable_state=trainable_state_dict(model), cfg=cfg, epoch=ep + 1, seed=seed,
                                val_macro_f1=best_f1, class_names=CLASS_NAMES), best_path)
            else:
                bad += 1
            torch.save(dict(trainable_state=trainable_state_dict(model), opt=opt.state_dict(), scaler=scaler.state_dict(),
                            epoch=ep, best_f1=best_f1, best_ep=best_ep, bad=bad, hist=hist, gstep=gstep, cfg=cfg), last_path)
            if bad >= cfg["patience"]:
                log(f"Early stopping at epoch {ep+1} (best epoch {best_ep})")
                break
            cleanup_gpu()

        # ---- final evaluation with best checkpoint ----
        ck = torch.load(best_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["trainable_state"], strict=False)
        vprob, vy = predict_loader(model, va_loader)
        tprob, ty = predict_loader(model, te_loader)
        vm, tm = compute_metrics(vy, vprob), compute_metrics(ty, tprob)
        write_predictions_csv(run_dir / "val_predictions.csv", val_samples, vy, vprob)
        write_predictions_csv(run_dir / "test_predictions.csv", test_samples, ty, tprob)
        plot_confusion(tm["confusion"], f"Test confusion ({tag or run_dir.name}, seed {seed})", run_dir / "confusion_test.png")
        trm = None
        if eval_train_at_end:
            trprob, try_ = predict_loader(model, make_loader(ImgDS(train_samples, build_eval_transform(cfg["img_size"]))))
            trm = compute_metrics(try_, trprob)
        # curves
        try:
            import pandas as pd
            pd.DataFrame(hist).to_csv(run_dir / "history.csv", index=False)
            fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
            ax[0].plot([h["train_loss"] for h in hist], label="train"); ax[0].plot([h["val_loss"] for h in hist], label="val")
            ax[0].set_title("Loss"); ax[0].legend()
            ax[1].plot([h["val_macro_f1"] for h in hist]); ax[1].set_title("Val macro-F1")
            fig.tight_layout(); fig.savefig(run_dir / "curves.png", dpi=150); plt.close(fig)
        except Exception as e:
            log(f"  curves skipped: {e}")
        summary = dict(tag=tag or run_dir.name, seed=seed, cfg=cfg, best_epoch=best_ep, epochs_run=len(hist),
                       params_total=tot, params_trainable=trn, train_minutes=(time.time() - t_start) / 60.0,
                       val=vm, test=tm, train=trm, adapter_mb=os.path.getsize(best_path) / 1e6)
        save_json(summary, sum_path)
        log(f"DONE {run_dir.name}: test acc {tm['accuracy']:.4f} macroF1 {tm['macro_f1']:.4f} "
            f"TD-recall {tm['tooth_disc_recall']:.4f} (best ep {best_ep})")
        del model, opt; cleanup_gpu()
        return summary
    finally:
        CFG.clear(); CFG.update(base_cfg)



def run_training(run_dir, seed, cfg_overrides=None, *args, **kw):
    """RTX-2050 safe wrapper: on CUDA out-of-memory, halves batch_size and doubles grad_accum (effective batch unchanged)
    and resumes from the last finished epoch (at most one epoch is lost)."""
    ov = dict(cfg_overrides or {})
    bs, acc = ov.get("batch_size", CFG["batch_size"]), ov.get("grad_accum", CFG["grad_accum"])
    for attempt in range(4):
        cleanup_gpu()
        try:
            return _run_training_impl(run_dir, seed, {**ov, "batch_size": bs, "grad_accum": acc}, *args, **kw)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower() or bs <= 2:
                raise
            bs, acc = bs // 2, acc * 2
            log(f"[OOM] CUDA out of memory -> retrying with batch_size={bs}, grad_accum={acc} (effective batch kept), resuming from last epoch")
    raise RuntimeError("Out of memory even at batch_size=2")


def gpu_temp_clock():
    try:
        import subprocess
        o = subprocess.run(["nvidia-smi", "--query-gpu=temperature.gpu,clocks.sm,power.draw", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5).stdout.strip().split(",")
        return f"{o[0].strip()}C, {o[1].strip()}MHz, {o[2].strip()}W"
    except Exception:
        return "n/a"


def mean_sd(vals):
    v = np.array(vals, dtype=float)
    return float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0


log(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE}")
if torch.cuda.is_available():
    log(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB)")


[paths] external raw     : C:\Users\R Y Z E N\Desktop\roboflow_dataset_merged\roboflow_dataset_merged
[paths] external BenGraham: C:\Users\R Y Z E N\Desktop\paper\roboflow_bengram_illumination
[11:37:07] torch 2.5.1+cu121 | torchvision 0.20.1+cu121 | device cuda
[11:37:07] GPU: NVIDIA GeForce RTX 2050 (4.29 GB)


In [2]:
import pandas as pd
from scipy.stats import chi2, binomtest
N_BOOT, BOOT_SEED, N_BOOT_AUC = 10000, 42, 2000

def fast_metrics(y, pred):
    cm = np.bincount(y * NUM_CLASSES + pred, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES).astype(float)
    tp, sup, pp = np.diag(cm), cm.sum(1), cm.sum(0)
    rec = np.divide(tp, sup, out=np.zeros(NUM_CLASSES), where=sup > 0); prec = np.divide(tp, pp, out=np.zeros(NUM_CLASSES), where=pp > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(NUM_CLASSES), where=(prec + rec) > 0)
    return dict(accuracy=tp.sum() / cm.sum(), macro_precision=prec.mean(), macro_recall=rec.mean(), macro_f1=f1.mean(),
                tooth_disc_recall=rec[CLASS_NAMES.index("Tooth_Discoloration")])

def boot_ci(y, pred, prob=None):
    rng = np.random.RandomState(BOOT_SEED); n = len(y); idxs = rng.randint(0, n, size=(N_BOOT, n))
    res = {k: [] for k in fast_metrics(y, pred)}
    for ix in idxs:
        m = fast_metrics(y[ix], pred[ix])
        for k in res: res[k].append(m[k])
    out = {k: (float(fast_metrics(y, pred)[k]), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for k, v in res.items()}
    if prob is not None:
        Y = label_binarize(y, classes=list(range(NUM_CLASSES))); aucs = []
        for ix in idxs[:N_BOOT_AUC]:
            try: aucs.append(roc_auc_score(Y[ix], prob[ix], average="macro", multi_class="ovr"))
            except Exception: pass
        out["macro_auc"] = (float(roc_auc_score(Y, prob, average="macro", multi_class="ovr")), float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5)))
    return out

def mcnemar(y, pa, pb):
    ca, cb = pa == y, pb == y; b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
    stat = (abs(b - c) - 1) ** 2 / (b + c) if (b + c) > 0 else 0.0
    return dict(a_only_correct=b, b_only_correct=c, chi2_cc=float(stat), p_cc=float(chi2.sf(stat, 1)) if (b + c) > 0 else 1.0,
                p_exact=float(binomtest(b, b + c, 0.5).pvalue) if (b + c) > 0 else 1.0)

def paired_boot(y, pa, pb):
    rng = np.random.RandomState(BOOT_SEED); n = len(y); da, df_ = [], []
    for _ in range(N_BOOT):
        ix = rng.randint(0, n, n); ma, mb = fast_metrics(y[ix], pa[ix]), fast_metrics(y[ix], pb[ix])
        da.append(ma["accuracy"] - mb["accuracy"]); df_.append(ma["macro_f1"] - mb["macro_f1"])
    f = lambda v: (float(np.mean(v)), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return dict(delta_accuracy=f(da), delta_macro_f1=f(df_))

def load_pred_csv(path):
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    y = np.array([CLASS_NAMES.index(r["true_label"]) for r in rows]); pr = np.array([CLASS_NAMES.index(r["pred_label"]) for r in rows])
    prob = np.array([[float(r[f"prob_{c}"]) for c in CLASS_NAMES] for r in rows]) if f"prob_{CLASS_NAMES[0]}" in rows[0] else None
    return y, pr, prob, [r["filename"] for r in rows] if "filename" in rows[0] else None

In [3]:
# ---- A. Bootstrap CIs: new model (primary seed) on internal test + external ----
report, rows = {}, []
ty, tp, tprob, _ = load_pred_csv(OUT_MAIN / f"seed_{PRIMARY_SEED}" / "test_predictions.csv")
sets = {"internal_test": (ty, tp, tprob)}
for cond in ("raw", "bengram"):
    p = OUT_EXT / f"seed_{PRIMARY_SEED}" / f"{cond}_zero_shot_predictions.csv"
    if p.exists():
        y, pr, prob, _ = load_pred_csv(p); sets[f"external_{cond}_zero_shot"] = (y, pr, prob)
for name, (y, pr, prob) in sets.items():
    ci = boot_ci(y, pr, prob); report[name] = ci
    for k, (pt, lo, hi) in ci.items(): rows.append(dict(set=name, metric=k, point=pt, ci_lo=lo, ci_hi=hi))
    print(name, {k: f"{v[0]:.4f} [{v[1]:.4f},{v[2]:.4f}]" for k, v in ci.items()})
pd.DataFrame(rows).to_csv(OUT_STATS / "bootstrap_ci_new_model.csv", index=False)
# per-class recall CIs
rc = []
for name, (y, pr, prob) in sets.items():
    rng = np.random.RandomState(BOOT_SEED)
    for ci_, cn in enumerate(CLASS_NAMES):
        m = np.where(y == ci_)[0]
        if len(m) == 0: continue
        bs = [(pr[rng.choice(m, len(m))] == ci_).mean() for _ in range(N_BOOT)]
        rc.append(dict(set=name, cls=cn, n=len(m), recall=(pr[m] == ci_).mean(), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5)))
rc = pd.DataFrame(rc); rc.to_csv(OUT_STATS / "per_class_recall_ci.csv", index=False); print(rc.round(4).to_string(index=False))

internal_test {'accuracy': '0.9858 [0.9787,0.9921]', 'macro_precision': '0.9855 [0.9767,0.9929]', 'macro_recall': '0.9476 [0.9169,0.9751]', 'macro_f1': '0.9630 [0.9377,0.9822]', 'tooth_disc_recall': '0.7308 [0.5500,0.8947]', 'macro_auc': '0.9995 [0.9990,0.9998]'}
external_raw_zero_shot {'accuracy': '0.8070 [0.7824,0.8307]', 'macro_precision': '0.8896 [0.8748,0.9038]', 'macro_recall': '0.8179 [0.8024,0.8330]', 'macro_f1': '0.7866 [0.7664,0.8063]', 'tooth_disc_recall': '0.1343 [0.0895,0.1838]', 'macro_auc': '0.9346 [0.9267,0.9429]'}
external_bengram_zero_shot {'accuracy': '0.8032 [0.7786,0.8269]', 'macro_precision': '0.8683 [0.8433,0.8906]', 'macro_recall': '0.8117 [0.7953,0.8276]', 'macro_f1': '0.7807 [0.7596,0.8009]', 'tooth_disc_recall': '0.1343 [0.0885,0.1841]', 'macro_auc': '0.9299 [0.9214,0.9383]'}
                       set                 cls   n  recall     lo     hi
             internal_test            Calculus  67  1.0000 1.0000 1.0000
             internal_test              

In [4]:
# ---- B. Paired comparison vs paper-1 OralVision-M4 (same images) ----
paired = {}
if M4_INTERNAL_CSV.exists():
    rws = list(csv.DictReader(open(M4_INTERNAL_CSV, encoding="utf-8")))
    col = [c for c in rws[0].keys() if c.lower().startswith("m4") and c.endswith("_pred")][0]
    m4y = np.array([CLASS_NAMES.index(r["true_label"]) for r in rws]); m4p = np.array([CLASS_NAMES.index(r[col]) for r in rws])
    assert len(m4y) == len(ty) and (m4y == ty).all(), "M4 internal CSV is not aligned with the new test predictions"
    paired["internal_test"] = (ty, tp, m4p)
else:
    log(f"[skip] {M4_INTERNAL_CSV} not found")
for cond, path in (("raw", M4_EXT_RAW_CSV), ("bengram", M4_EXT_BG_CSV)):
    key = f"external_{cond}_zero_shot"
    if path.exists() and key in sets:
        m4y, m4p, _, m4f = load_pred_csv(path); y, pr, _ = sets[key]
        nf = [os.path.basename(s[0]) for s in collect_external_samples(str(RAW_DATA_DIR if cond == "raw" else BENGRAM_DATA_DIR))]
        assert len(m4y) == len(y) and (m4y == y).all() and m4f == nf, f"M4 external ({cond}) not aligned with new predictions"
        paired[key] = (y, pr, m4p)
    else:
        log(f"[skip] paired comparison for {key}")
prow = []
for name, (y, pa, pb) in paired.items():
    mc, pbt = mcnemar(y, pa, pb), paired_boot(y, pa, pb); ma, mb = fast_metrics(y, pa), fast_metrics(y, pb)
    prow.append(dict(set=name, new_acc=ma["accuracy"], m4_acc=mb["accuracy"], new_f1=ma["macro_f1"], m4_f1=mb["macro_f1"],
                     new_TD_recall=ma["tooth_disc_recall"], m4_TD_recall=mb["tooth_disc_recall"], new_only_correct=mc["a_only_correct"],
                     m4_only_correct=mc["b_only_correct"], mcnemar_p_cc=mc["p_cc"], mcnemar_p_exact=mc["p_exact"],
                     d_acc=f"{pbt['delta_accuracy'][0]:+.4f} [{pbt['delta_accuracy'][1]:+.4f},{pbt['delta_accuracy'][2]:+.4f}]",
                     d_macroF1=f"{pbt['delta_macro_f1'][0]:+.4f} [{pbt['delta_macro_f1'][1]:+.4f},{pbt['delta_macro_f1'][2]:+.4f}]"))
if prow:
    pt = pd.DataFrame(prow); print(pt.to_string(index=False)); pt.to_csv(OUT_STATS / "paired_vs_paper1_M4.csv", index=False)

[11:37:36]   [NOTE] duplicate folder for 'Mouth_Ulcer' ('Mouth_Ulcer' vs 'Mouth Ulcer') -> skipped duplicate
[11:37:36]   [NOTE] duplicate folder for 'Tooth_Discoloration' ('Tooth_Discoloration' vs 'Tooth Discoloration') -> skipped duplicate
                       set  new_acc   m4_acc   new_f1    m4_f1  new_TD_recall  m4_TD_recall  new_only_correct  m4_only_correct  mcnemar_p_cc  mcnemar_p_exact                     d_acc                 d_macroF1
             internal_test 0.985771 0.995257 0.963023 0.984145       0.730769      0.884615                 2               14      0.005960         0.004181 -0.0095 [-0.0158,-0.0040] -0.0214 [-0.0419,-0.0054]
    external_raw_zero_shot 0.807001 0.833491 0.786576 0.821061       0.134328      0.278607                18               46      0.000738         0.000617 -0.0266 [-0.0416,-0.0114] -0.0345 [-0.0538,-0.0150]
external_bengram_zero_shot 0.803217 0.812677 0.780696 0.789223       0.134328      0.159204                17               27  

In [5]:
# ---- C. Does test-time adaptation help? (paired vs zero-shot, primary seed) ----
trows = []
for cond in ("raw", "bengram"):
    base = OUT_EXT / f"seed_{PRIMARY_SEED}" / f"{cond}_zero_shot_predictions.csv"
    if not base.exists(): continue
    y, p0, _, _ = load_pred_csv(base)
    for mode in ("recenter_mean", "recenter_meanstd", "tent_online", "tent_episodic"):
        pp = OUT_EXT / f"seed_{PRIMARY_SEED}" / f"{cond}_{mode}_predictions.csv"
        if not pp.exists(): continue
        _, p1, _, _ = load_pred_csv(pp); mc, pb = mcnemar(y, p1, p0), paired_boot(y, p1, p0); m1, m0 = fast_metrics(y, p1), fast_metrics(y, p0)
        trows.append(dict(condition=cond, mode=mode, acc=m1["accuracy"], zs_acc=m0["accuracy"], macroF1=m1["macro_f1"], zs_macroF1=m0["macro_f1"],
                          TD_recall=m1["tooth_disc_recall"], zs_TD_recall=m0["tooth_disc_recall"], mcnemar_p_cc=mc["p_cc"],
                          d_macroF1=f"{pb['delta_macro_f1'][0]:+.4f} [{pb['delta_macro_f1'][1]:+.4f},{pb['delta_macro_f1'][2]:+.4f}]"))
if trows:
    tt = pd.DataFrame(trows); print(tt.round(4).to_string(index=False)); tt.to_csv(OUT_STATS / "tta_vs_zero_shot_paired.csv", index=False)

condition             mode    acc  zs_acc  macroF1  zs_macroF1  TD_recall  zs_TD_recall  mcnemar_p_cc                 d_macroF1
      raw    recenter_mean 0.8013  0.8070   0.7745      0.7866     0.1095        0.1343        0.1489 -0.0121 [-0.0221,-0.0026]
      raw recenter_meanstd 0.8004  0.8070   0.7734      0.7866     0.1095        0.1343        0.0961 -0.0132 [-0.0235,-0.0037]
      raw      tent_online 0.8051  0.8070   0.7836      0.7866     0.1244        0.1343        0.4795 -0.0030 [-0.0077,+0.0000]
      raw    tent_episodic 0.8070  0.8070   0.7866      0.7866     0.1343        0.1343        1.0000 +0.0000 [+0.0000,+0.0000]
  bengram    recenter_mean 0.7956  0.8032   0.7669      0.7807     0.0995        0.1343        0.0614 -0.0138 [-0.0247,-0.0040]
  bengram recenter_meanstd 0.7947  0.8032   0.7660      0.7807     0.0995        0.1343        0.0265 -0.0147 [-0.0257,-0.0050]
  bengram      tent_online 0.8004  0.8032   0.7763      0.7807     0.1244        0.1343        0.3711 -0

In [6]:
# ---- D. Complexity + latency (same machine, paper-1 model rebuilt with random weights: only architecture matters here) ----
from torchvision.models import efficientnet_b0
from torch.utils.flop_counter import FlopCounterMode

class CoordAtt(nn.Module):
    def __init__(self, inp, oup, reduction=32):
        super().__init__()
        self.pool_h, self.pool_w = nn.AdaptiveAvgPool2d((None, 1)), nn.AdaptiveAvgPool2d((1, None)); mip = max(8, inp // reduction)
        self.conv1, self.bn1, self.act = nn.Conv2d(inp, mip, 1), nn.BatchNorm2d(mip), nn.Hardswish()
        self.conv_h, self.conv_w, self.sigmoid = nn.Conv2d(mip, oup, 1), nn.Conv2d(mip, oup, 1), nn.Sigmoid()
    def forward(self, x):
        idt = x; n, c, h, w = x.size(); xh = self.pool_h(x); xw = self.pool_w(x).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([xh, xw], dim=2)))); xh, xw = torch.split(y, [h, w], dim=2)
        return idt * self.sigmoid(self.conv_h(xh)) * self.sigmoid(self.conv_w(xw.permute(0, 1, 3, 2)))

class EfficientNetCoordAtt(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__(); b = efficientnet_b0(weights=None); self.features = b.features
        self.coordatt, self.avgpool, self.classifier = CoordAtt(1280, 1280), nn.AdaptiveAvgPool2d(1), nn.Linear(1280, num_classes)
    def forward(self, x): return self.classifier(torch.flatten(self.avgpool(self.coordatt(self.features(x))), 1))

new_model, ck = load_model_from_ckpt(OUT_MAIN / f"seed_{PRIMARY_SEED}" / "best.pt")
m4_model = EfficientNetCoordAtt().to(DEVICE).eval()

def flops_g(model):
    m = copy.deepcopy(model).cpu().eval()
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        m(torch.randn(1, 3, 224, 224))
    return fc.get_total_flops() / 1e9

@torch.no_grad()
def latency(model, bs, device, amp, n=60, warm=10):
    m = model.to(device).eval(); x = torch.randn(bs, 3, 224, 224, device=device); ts = []
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    for i in range(warm + n):
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        with autocast(device_type=device.type, enabled=amp and device.type == "cuda"): m(x)
        if device.type == "cuda": torch.cuda.synchronize()
        if i >= warm: ts.append((time.perf_counter() - t0) * 1000)
    peak = torch.cuda.max_memory_allocated() / 1e6 if device.type == "cuda" else float("nan")
    return float(np.median(ts)), float(np.percentile(ts, 95)), peak

rows = []
for name, mdl in (("OralViT-P2 (DINOv2 ViT-S/14 + LoRA)", new_model), ("OralVision-M4 (EffNet-B0 + CoordAtt)", m4_model)):
    tot, trn = count_params(mdl)
    fl_model = OralViT(ck["cfg"], eager=True) if mdl is new_model else mdl   # eager attention so attention matmuls are counted
    row = dict(model=name, params_M=tot / 1e6, trainable_at_finetune_M=trn / 1e6, fp32_size_MB=tot * 4 / 1e6, GFLOPs_224=flops_g(fl_model))
    if DEVICE.type == "cuda":
        for bs in (1, 32):
            try:
                med, p95, peak = latency(mdl, bs, DEVICE, True)
            except RuntimeError:
                cleanup_gpu(); med = p95 = peak = float("nan"); log(f"[OOM] latency bs{bs} skipped")
            row[f"gpu_fp16_bs{bs}_ms_median"] = med; row[f"gpu_fp16_bs{bs}_ms_p95"] = p95; row[f"gpu_peak_MB_bs{bs}"] = peak
    med, p95, _ = latency(mdl, 1, torch.device("cpu"), False, n=20, warm=3); row["cpu_bs1_ms_median"] = med; row["cpu_bs1_ms_p95"] = p95
    mdl.to("cpu"); rows.append(row)
row0 = rows[0]; row0["adapter_checkpoint_MB"] = os.path.getsize(OUT_MAIN / f"seed_{PRIMARY_SEED}" / "best.pt") / 1e6
cx = pd.DataFrame(rows); print(cx.T.to_string()); cx.to_csv(OUT_STATS / "complexity_latency.csv", index=False)
log("NOTE: desktop RTX 2050 / CPU timings only — no mobile-device deployment claim. FLOPs = matmul/conv ops only, counted as 2 x multiply-accumulate (same convention for both models).")

                                                           0                                     1
model                    OralViT-P2 (DINOv2 ViT-S/14 + LoRA)  OralVision-M4 (EffNet-B0 + CoordAtt)
params_M                                            22.74983                              4.171514
trainable_at_finetune_M                             0.693254                              4.171514
fp32_size_MB                                        90.99932                             16.686056
GFLOPs_224                                         12.399501                              0.771952
gpu_fp16_bs1_ms_median                               36.5406                              23.56725
gpu_fp16_bs1_ms_p95                                 51.85374                             28.571515
gpu_peak_MB_bs1                                   124.308992                             36.729344
gpu_fp16_bs32_ms_median                            112.64835                              37.74775
gpu_fp16_b